# Estimated foot traffic

In [24]:
import pandas as pd
import matplotlib.pyplot as plt

# Change the path
df = pd.read_excel(r'E:\historical_density_8232020-10112023.xlsm')
df.head()

,Building Address,Building Type,building_code,Building_Floor,building_name,Capacity,date,Floor,Hour 2,Percentage,Percentage of Capacity Used,Population,Population % difference from Capacity
0,NaN,NaN,514A,514A_1,NaN,0.0,2020-08-23,1,17:00:00,0.0,NaN,2.0,NaN
1,NaN,NaN,514A,514A_1,NaN,0.0,2020-08-23,1,18:00:00,0.0,NaN,2.0,NaN
2,NaN,NaN,514A,514A_1,NaN,0.0,2020-08-25,1,17:00:00,0.0,NaN,4.0,NaN
3,NaN,NaN,514A,514A_1,NaN,0.0,2020-08-26,1,11:00:00,0.0,NaN,4.0,NaN
4,NaN,NaN,514A,514A_1,NaN,0.0,2020-08-26,1,12:00:00,0.0,NaN,2.0,NaN


Clean the data:

In [25]:
# Removing rows where 'Building Address' is NaN
cleaned_df = df.dropna(subset=['Building Address'])

# Group the data by 'Building Address', 'date', and 'Hour 2' to find the total population for each address at each date and time
grouped_df = cleaned_df.groupby(['Building Address', 'date', 'Hour 2'])['Population'].sum().reset_index()

# Summary of this grouped data
summary = grouped_df.groupby('Building Address')['Population'].describe()
summary

,count,mean,std,min,25%,50%,75%,max
Building Address,,,,,,,,
610 ALBANY STREET - GARAGE,15278.0,41.564865,41.605880,1.0,7.0,27.0,67.0,364.0
710 ALBANY STREET - GARAGE,11254.0,42.691310,37.188087,1.0,15.0,32.0,57.0,204.0
80 EAST CONCORD STREET - A-BLDG,21140.0,96.438978,109.271954,1.0,18.0,47.0,147.0,609.0
925 COMMONWEALTH AVENUE,10480.0,24.263263,102.084971,1.0,2.0,3.0,7.0,1859.0


Calculate the average number of people flow in the same month of each year

In [29]:
import numpy as np

## Calculate people flow through Population for 5 specific days each month
# Sum visitors for 5 random days in a given month for each building
'''
def sum_visitors_random_days(df, month):
    df_month = df[df['date'].dt.month == month]
    df_month = df_month[df_month['date'].dt.year == 2020]
    
    sum_visitors = df_month.groupby(['Building Address', df_month['date'].dt.day])['Population'].sum()
    
    results = {}
    for building in sum_visitors.index.levels[0]:
        building_data = sum_visitors[building]
        if len(building_data) >= 5:
            selected_days = building_data.sample(5, random_state=1)
            results[building] = selected_days.sum()
        else:
            results[building] = None 
    
    return results

# Calculate the total visitors for the months
months = [8, 9, 10]
visitors_per_month = {month: sum_visitors_random_days(df, month) for month in months}

visitors_df = pd.DataFrame.from_dict(visitors_per_month, orient='index').T
visitors_df.reset_index(inplace=True)
visitors_df.rename(columns={'index': 'Building Address', 8: 'August', 9: 'September', 10: 'October'}, inplace=True)

visitors_df
'''

def avg_daily_visitors_per_month(df):
    df['date'] = pd.to_datetime(df['date'])
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day

    daily_sum = df.groupby(['Building Address', 'year', 'month', 'day'])['Population'].sum().reset_index()
  
    monthly_avg = daily_sum.groupby(['Building Address', 'month'])['Population'].mean().round(2)
 
    result_df = monthly_avg.unstack(level=-1)

    month_names = {i: pd.date_range('2020-{}'.format(i), periods=1, freq='M').month_name()[0] for i in range(1,13)}
    result_df = result_df.rename(columns=month_names)
    
    return result_df

avg_visitors_df = avg_daily_visitors_per_month(df)
avg_visitors_df


month,January,February,March,April,May,June,July,August,September,October,November,December
Building Address,,,,,,,,,,,,
610 ALBANY STREET - GARAGE,530.20,593.94,725.01,700.25,708.26,655.01,443.19,539.55,521.58,647.89,606.34,497.91
710 ALBANY STREET - GARAGE,541.13,752.64,791.92,766.25,697.57,614.08,605.27,790.00,592.55,638.38,633.50,616.86
80 EAST CONCORD STREET - A-BLDG,1119.85,1304.07,1849.41,2280.44,1759.16,1733.22,1758.22,1979.43,2043.12,1750.82,1982.89,1726.39
925 COMMONWEALTH AVENUE,242.74,215.73,187.72,147.42,476.86,125.31,61.51,122.21,157.29,625.81,255.11,438.44


We will not consider all data in the current excel table which street address is NaN. The statistical range is from January to December 2020 to 2023. There are currently four addresses in the excel table, and we have calculate the average number of people flow through Population for the same month of each year.